# Modeling walkthrough: popularity, item-item CF, and ALS

A narrative tour of the three models compared in this project, weakest to
strongest. The actual pipeline lives in `src/` (`python -m src.evaluate tune`
and `python -m src.evaluate test` are what produced `reports/results.md`); this
notebook re-derives the same steps on a smaller scale to make the mechanics
visible.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src.config import PROCESSED_DIR, ALS_ALPHA, ALS_FACTORS, ALS_REG, ALS_ITERATIONS
from src.evaluate import build_id_maps, to_sparse, rank_all_users, _seen_and_relevant
from src.models.popularity import fit_popularity
from src.models.item_knn import fit_item_knn, score_users
from src.models.als import ImplicitALS

train = pd.read_csv(PROCESSED_DIR / 'train.csv')
val = pd.read_csv(PROCESSED_DIR / 'val.csv')
user_ids, item_ids, uidx, iidx = build_id_maps(train, val)
n_users, n_items = len(user_ids), len(item_ids)
print(f'{n_users:,} users, {n_items:,} items')

6,038 users, 3,520 items


## 1. Popularity: the mandatory baseline

Everyone gets the same ranking, ordered by how many users interacted with each
item in training. Non-personalised, essentially free to compute, and on a
head-heavy catalog like this one it's embarrassingly competitive; any
personalised model that can't beat it isn't paying for itself.

In [2]:
train_binary = to_sparse(train, uidx, iidx, n_users, n_items)
pop_scores = fit_popularity(train_binary)
top_pop_idx = np.argsort(-pop_scores)[:5]
print('Most popular movies in the training set:')
for i in top_pop_idx:
    print(f'  item_id={item_ids[i]}  interactions={int(pop_scores[i])}')

Most popular movies in the training set:
  item_id=2858  interactions=2639
  item_id=260  interactions=2435
  item_id=1196  interactions=2358
  item_id=1198  interactions=2127
  item_id=2028  interactions=2108


## 2. Item-item collaborative filtering

Items become binary vectors over users; cosine similarity between those
vectors (truncated to the top-100 neighbours per item) gives a sparse
similarity matrix. A user's score for an item is the sum of similarities
between that item and everything in their history, which makes every
recommendation decomposable into the history items that drove it (see the
"why" panel in `app.py`).

In [3]:
sim = fit_item_knn(train_binary, top_k=100)
print(f'Similarity matrix: {sim.shape}, {sim.nnz:,} nonzero entries '
      f'({sim.nnz / (sim.shape[0] * sim.shape[1]) * 100:.2f}% dense)')

knn_scores = score_users(train_binary, sim)
print('Item-item CF score matrix shape:', knn_scores.shape)

Similarity matrix: (3520, 3520), 348,030 nonzero entries (2.81% dense)


Item-item CF score matrix shape: (6038, 3520)


## 3. ALS: implicit-feedback matrix factorization

From scratch, following Hu, Koren & Volinsky (2008). Every user gets a factor
vector, every item gets a factor vector, and observed positives are weighted
by a confidence that scales with the rating value:

$$c_{ui} = 1 + \alpha \, r_{ui}$$

The closed-form per-user update, solved with a Cholesky factorisation each
sweep, is derived in full in `README.md`; `src/models/als.py` is the ~30-line
implementation of it.

In [4]:
train_ratings = to_sparse(train, uidx, iidx, n_users, n_items, value_col='rating')
als = ImplicitALS(factors=ALS_FACTORS, reg=ALS_REG, alpha=ALS_ALPHA, iterations=ALS_ITERATIONS, random_state=42)
als.fit(train_ratings)
print(f'Fitted ALS: {als.X.shape[1]} factors, {als.X.shape[0]:,} users, {als.Y.shape[0]:,} items')

Fitted ALS: 64 factors, 6,038 users, 3,520 items


## Comparing all three on the validation slice

Same ranking metrics as the final test evaluation (`Recall@10`, `NDCG@10`,
`MAP@10`, `Coverage@10`), just run here on validation instead of the held-out
test set.

In [5]:
eval_users = np.array(sorted({uidx[u] for u in val['user_id'].unique()}))
seen_by_user, relevant_by_user = _seen_and_relevant(train, val, uidx, iidx)

pop_metrics = rank_all_users(lambda u: pop_scores, eval_users, seen_by_user, relevant_by_user, n_items)
knn_metrics = rank_all_users(lambda u: knn_scores[u], eval_users, seen_by_user, relevant_by_user, n_items)
als_metrics = rank_all_users(lambda u: als.score(u), eval_users, seen_by_user, relevant_by_user, n_items)

for name, m in [('popularity', pop_metrics), ('item-knn', knn_metrics), ('als', als_metrics)]:
    print(f"{name:12s} recall={m['recall'].mean():.4f}  ndcg={m['ndcg'].mean():.4f}  "
          f"map={m['map'].mean():.4f}  coverage={m['coverage']:.1%}")

popularity   recall=0.0491  ndcg=0.0413  map=0.0195  coverage=3.3%
item-knn     recall=0.0695  ndcg=0.0583  map=0.0280  coverage=9.9%
als          recall=0.0876  ndcg=0.0698  map=0.0324  coverage=31.2%


## Takeaway

ALS should come out ahead on accuracy *and* reach further into the catalog's
tail (higher coverage) than either baseline, at the cost of being the most
expensive to train and the hardest to explain per-recommendation. Item-item CF
sits in between: cheaper than ALS, more personalised than popularity, and
naturally explainable. See `reports/results.md` for the real numbers on the
held-out test set, and `reports/hyperparam_sensitivity.md` for how each ALS
hyperparameter was chosen.